# ASR Pipeline — interactive exploration

Run each stage of the pipeline on its own. Tweak knobs, re-run a stage, accept its
output, move on to the next.

```
Diarization → Routing → SE (solo) → SS (overlap) → Assembly → Transcription
```

**Workflow:**

1. Cell 1: imports + default `INPUT_PATH`.
2. Cell 2 (optional): drag an audio file onto the widget. Updates `INPUT_PATH`.
3. Cell 3: build `cfg`, construct `pipeline`, load audio into `ctx`.
4. Cells 4 onwards — one per stage. Each cell has a knob block at the top, a
   `pipeline.run_stage(...)` call, and an inspection block. Re-run the cell as
   many times as you like; the stage's model stays loaded between runs.
5. When you switch to a different stage, the previous model is freed and the
   new one loads (one-model-on-GPU invariant — same as the POC).
6. Final cell: `pipeline.unload()` releases the last loaded model.

**Prerequisites:** `pyannote.audio`, `whisper`, `speechbrain`, MP-SENet checkpoint
+ `config.json`, the SepFormer checkpoint pointed to by `default.yaml`.

In [ ]:
from IPython.display import HTML, display
display(HTML("<style>audio{width:100% !important;}</style>"))

# --- Imports + path bootstrapping ---
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "asr":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)   # so relative checkpoint paths resolve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Audio, display

# Edit for a different input recording (overridden by the drag-and-drop in cell 2).
# INPUT_PATH = "/home/user/datasets/clarin_gotowy/gotowy/442dd69e.wav"
INPUT_PATH = "/home/user/datasets/clarin_gotowy/gotowy/e14aa22f.wav"
# INPUT_PATH = "/home/user/datasets/LibriCSS_2spk/record/segments/OV40_session9_seg4.wav"
# INPUT_PATH = "/home/user/datasets/EdAcc/edacc/audios/EAEC-C02.wav"
# INPUT_PATH = "/home/user/datasets/clarin_all_2speaker_fragments/fragments/026eafb1__seg00.wav"

print("Project root:", PROJECT_ROOT)
print("Input       :", INPUT_PATH, "(exists:", Path(INPUT_PATH).exists(), ")")

In [ ]:
# --- Drag-and-drop audio loader (optional) -------------------------------
# Drop an audio file onto the widget (or click to pick one). On upload,
# the file is saved to /tmp and `INPUT_PATH` is updated. If you don't use
# the widget, the path set in cell 1 stays in effect.
import tempfile
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg,.m4a",
    multiple=False,
    description="Drop audio here",
    layout=widgets.Layout(width="320px"),
)
status = widgets.HTML(value=f"<i>Using path from cell 1: <code>{INPUT_PATH}</code></i>")


def _on_upload(change):
    global INPUT_PATH
    val = uploader.value
    if not val:
        return
    if isinstance(val, dict):
        name = next(iter(val))
        content = val[name]["content"]
    else:
        item = val[0]
        name = item["name"]
        content = item["content"]
    out = Path(tempfile.gettempdir()) / f"upload_{name}"
    out.write_bytes(bytes(content))
    INPUT_PATH = str(out)
    status.value = (
        f"<b>Uploaded:</b> <code>{name}</code> &rarr; "
        f"<code>{INPUT_PATH}</code> &nbsp;({len(content)/1024:.1f} KiB)"
    )


uploader.observe(_on_upload, names="value")
display(widgets.VBox([uploader, status]))

In [ ]:
# --- Build config, construct pipeline, load audio ---
from asr_pipeline import Pipeline
from asr_pipeline.config import load_pipeline_config_from_yaml

cfg = load_pipeline_config_from_yaml("asr_pipeline/configs/frcrn_vadstrict.yaml")

# Optional: persist intermediates to disk in addition to the in-memory `ctx`.
# cfg.artifact_dir = "runs/notebook"
# cfg.spill_intermediate = True
cfg.__post_init__()

pipeline = Pipeline(cfg)
ctx = pipeline.load_audio(INPUT_PATH)
print(pipeline)
print(f"Audio loaded: {ctx.audio.shape[0]/ctx.sample_rate:.2f}s @ {ctx.sample_rate}Hz")

## Stage 1 — Diarization

Per-speaker bars + red highlights for pyannote-detected overlaps.

Re-run this cell after editing the knobs to redo diarization without touching
other stages. The pyannote model stays loaded between runs.

In [ ]:
# --- Knobs ---
# cfg.diarization.num_speakers = 2

pipeline.run_stage("diarization", ctx)


# --- Inspect ---
def plot_diarization(seg_df, ovl_df, total_dur, title="Diarization"):
    if len(seg_df) == 0:
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.text(0.5, 0.5, "No speakers detected", ha="center", va="center")
        return fig
    speakers = sorted(seg_df["speaker"].unique())
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}
    fig, ax = plt.subplots(figsize=(16, max(2.5, len(speakers) + 1)))
    for _, row in seg_df.iterrows():
        ax.barh(spk_to_y[row["speaker"]], row["duration"], left=row["start"],
                height=0.6, color=spk_color[row["speaker"]], alpha=0.85)
    for _, row in ovl_df.iterrows():
        ax.axvspan(row["start"], row["end"], color="#e74c3c", alpha=0.25)
    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(speakers, fontweight="bold")
    ax.set_xlim(0, total_dur); ax.set_xlabel("Time (s)"); ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    handles = [mpatches.Patch(color=spk_color[s], label=s) for s in speakers]
    handles.append(mpatches.Patch(color="#e74c3c", alpha=0.4, label="overlap"))
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title, fontweight="bold"); plt.tight_layout()
    return fig

plot_diarization(
    ctx.diarization.segments_df,
    ctx.diarization.overlaps_df,
    ctx.diarization.total_duration_s,
)
plt.show()
print(f"{len(ctx.diarization.segments_df)} segments, "
      f"{len(ctx.diarization.overlaps_df)} overlaps")
display(ctx.diarization.segments_df.head(10))

## Stage 2 — Routing (overlap selection)

Decide which intervals get sent to SepFormer: filter pyannote's overlap
timeline by `min_overlap_dur`, then merge regions closer than `merge_gap`
so SepFormer sees one contiguous call per cluster.

Per-speaker solos are **not** produced here; the assembler in Stage 4
derives them on the fly by subtracting these overlap regions from
pyannote's per-speaker segments.

In [ ]:
# --- Knobs ---
# cfg.routing.min_overlap_dur = 0.10
# cfg.routing.merge_gap       = 0.20

pipeline.run_stage("routing", ctx)


# --- Inspect: overlay diarization (background) + routing overlap regions (red) ---
def plot_routing(seg_df, overlap_regions, total_dur, title="Routing — overlap selection"):
    """Per-speaker pyannote segments + red strips for ctx.overlap_regions.

    No solo bars — the assembler derives per-speaker solos on the fly by
    subtracting overlap_regions from pyannote segments. The visualisation
    shows: (a) what pyannote saw per speaker, (b) which slices will be
    sent to SepFormer.
    """
    if len(seg_df) == 0:
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.text(0.5, 0.5, "No speakers detected", ha="center", va="center")
        return fig
    speakers = sorted(seg_df["speaker"].unique())
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    spk_to_label = {s: chr(ord("A") + i) for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}

    fig, ax = plt.subplots(figsize=(16, max(2.5, len(speakers) + 1)))

    # Foreground: pyannote per-speaker segments (unpadded).
    for _, row in seg_df.iterrows():
        ax.barh(
            spk_to_y[row["speaker"]],
            row["duration"],
            left=row["start"],
            height=0.6,
            color=spk_color[row["speaker"]],
            alpha=0.85,
        )

    # Background: overlap regions sent to SepFormer (post-merge).
    for s, e in overlap_regions:
        ax.axvspan(s, e, color="#e74c3c", alpha=0.25)

    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(
        [f"{spk_to_label[s]}: {s}" for s in speakers], fontweight="bold"
    )
    ax.set_xlim(0, total_dur)
    ax.set_xlabel("Time (s)")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3, linestyle="--")

    handles = [
        mpatches.Patch(color=spk_color[s], label=spk_to_label[s])
        for s in speakers
    ]
    handles.append(mpatches.Patch(color="#e74c3c", alpha=0.4, label="→ SepFormer"))
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title, fontweight="bold")
    plt.tight_layout()
    return fig


plot_routing(
    ctx.diarization.segments_df,
    ctx.overlap_regions,
    ctx.diarization.total_duration_s,
)
plt.show()

total_overlap_s = sum(e - s for s, e in ctx.overlap_regions)
print(f"speakers          : {ctx.speakers}")
print(f"overlap regions   : {len(ctx.overlap_regions)} "
      f"({total_overlap_s:.2f}s total → SepFormer)")
for i, (s, e) in enumerate(ctx.overlap_regions):
    print(f"  #{i:2d}  {s:6.2f} - {e:6.2f}  ({e - s:.2f}s)")


## Stage 3a — Full-recording enhancement 

One enhancement pass over the whole input. Recordings longer than
`max_segment_length_s` (default 8 s) are processed via Hann overlap-add.
The result lives in `ctx.enhanced_full` and is sliced per speaker at
assembly time.

Note: SepFormer in Stage 3b runs on the *original* `ctx.audio`, not on
this enhanced version — enhancement model is a denoiser, not a separator, and
tends to suppress the quieter speaker in overlapping speech.

The model loads on the first run of this cell; subsequent runs
reuse the same loaded model.

In [ ]:
# --- Knobs ---
# cfg.enhancement.max_segment_length_s = 8.0   # solos longer than this are chunked (mpsenet only)
# cfg.enhancement.backend = "mossformer_gan_se_16k"   # mpsenet / frcrn_se_16k / mossformer_gan_se_16k / mossformer2_se_48k

pipeline.run_stage("enhancement", ctx)


# --- Inspect: full recording before vs after enhancement ---
sr = ctx.sample_rate
raw = ctx.audio
enhanced = ctx.enhanced_full
t = np.arange(len(raw)) / sr

fig, axes = plt.subplots(2, 1, figsize=(16, 4), sharex=True, sharey=True)
axes[0].plot(t, raw, linewidth=0.4, color="C0")
axes[0].set_ylabel("raw")
axes[0].grid(alpha=0.3)
axes[1].plot(t, enhanced, linewidth=0.4, color="C1")
axes[1].set_ylabel("enhanced")
axes[1].set_xlabel("Time (s)")
axes[1].grid(alpha=0.3)
axes[0].set_title("Full-recording enhancement", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"backend  : {cfg.enhancement.backend}")
print(f"duration : {len(raw)/sr:.2f}s @ {sr}Hz")
if cfg.enhancement.backend == "mpsenet":
    long_input = len(raw) / sr > cfg.enhancement.max_segment_length_s
    print(f"chunking : > {cfg.enhancement.max_segment_length_s:.1f}s → "
          f"Hann overlap-add ({'yes' if long_input else 'no — one shot'})")
else:
    print("chunking : handled internally by backend")

print("\nraw:")
display(Audio(raw, rate=sr))
print("enhanced:")
display(Audio(enhanced, rate=sr))

## Stage 3b — Overlap separation (SepFormer + VAD)

For each overlap region:

- top: mixture (16 kHz)
- middle two: separator outputs s1 / s2 — raw (translucent) vs VAD-gated
- 4th panel: per-stream binary VAD **mask** (after soft threshold + dilation)
- 5th panel: raw silero **VAD probabilities** per 32 ms frame, with
  horizontal lines at `vad_threshold` (upper, "definitely speech") and
  `vad_soft_threshold` (Schmitt-trigger lower bound) — read off which
  frames cross which cutoff to see *why* the mask came out the way it did
- green tint: the **emit region** that the assembler will splice into the
  per-speaker stream (controlled by `cfg.separation.seam_mode`)

A one-line summary is printed for every overlap, but detailed plots + audio
are only rendered for the first `MAX_DETAILED_PLOTS` (default 20). Many
inline outputs in a single cell destabilise the VS Code WSL notebook bridge.

Try editing one of the knobs (e.g. switch `context_window_mode` to `fixed_pad`)
and re-running this cell to see how the pad/emit regions move.

In [ ]:
# --- Knobs ---
# cfg.separation.context_window_mode       = "expand_to_chunk"    # expand_to_chunk | fixed_pad | none
# cfg.separation.context_pad_seconds       = 1.0
# cfg.separation.min_fragment_length_s     = 4.0                  # floor for padded-window length
# cfg.separation.seam_mode                 = "snap_to_silence"    # zero_crossing | overlap_boundary | snap_to_silence
# cfg.separation.seam_search_radius_s      = 0.05                 # zero-crossing nudge radius
# cfg.separation.snap_silence_max_extend_s = 0.30                 # snap_to_silence: max outward extension (s)
# cfg.separation.volume_normalization      = "sum_equals_mix"     # sum_equals_mix | none
# cfg.separation.vad_threshold             = 0.25                 # upper ("definitely speech")
# cfg.separation.vad_soft_threshold        = 0.10                 # Schmitt-trigger lower (0 = disable)
# cfg.separation.vad_attack_frames         = 1                    # 32 ms each
# cfg.separation.vad_release_frames        = 1

# Inspection cap. The 6-subplot figure + 3 audio players per overlap means
# ~7 rich outputs per overlap region — on a 15-minute recording with 50+
# overlaps that's 350+ outputs in one cell, which makes the VS Code WSL
# notebook bridge unstable ("reconnecting to remote authority" / hung kernel
# on the next cell). We always print a one-line summary for every overlap,
# but only render plots + audio for the first MAX_DETAILED_PLOTS.
MAX_DETAILED_PLOTS = 20

pipeline.run_stage("separation", ctx)


# --- Inspect ---
sr = ctx.sample_rate
if not ctx.overlap_separated:
    print("No overlap regions in this recording.")
else:
    # 1) Compact summary table for every overlap (cheap, always shown).
    print(f"{len(ctx.overlap_separated)} overlap region(s):")
    print(f"  {'idx':>4} {'orig (s)':>16} {'pad (s)':>16} {'emit (s)':>16}  "
          f"{'chunked':>7}  {'vol':>5}")
    for ovl in ctx.overlap_separated:
        print(f"  #{ovl['idx']:>3} "
              f"[{ovl['start']:6.2f},{ovl['end']:6.2f}] "
              f"[{ovl['pad_start']:6.2f},{ovl['pad_end']:6.2f}] "
              f"[{ovl['emit_start']:6.2f},{ovl['emit_end']:6.2f}]  "
              f"{str(ovl['chunked']):>7}  {ovl['volume_scale']:5.2f}")

    # 2) Detailed plot + audio only for the first MAX_DETAILED_PLOTS.
    n_total = len(ctx.overlap_separated)
    to_render = ctx.overlap_separated[:MAX_DETAILED_PLOTS]
    if n_total > len(to_render):
        print(f"\nRendering plots + audio for first {len(to_render)} of {n_total} "
              f"overlaps (raise MAX_DETAILED_PLOTS to see more).")

    # pyannote diarization (used for the narrow top band in each plot).
    diar_df_full = ctx.diarization.segments_df if ctx.diarization is not None else None

    for ovl in to_render:
        print(f"\nOverlap #{ovl['idx']} — "
              f"orig=[{ovl['start']:.2f}, {ovl['end']:.2f}]  "
              f"pad=[{ovl['pad_start']:.2f}, {ovl['pad_end']:.2f}]  "
              f"emit=[{ovl['emit_start']:.2f}, {ovl['emit_end']:.2f}]  "
              f"chunked={ovl['chunked']}  vol_scale={ovl['volume_scale']:.3f}")

        t = np.arange(len(ovl["mix"])) / sr
        window_dur = len(ovl["mix"]) / sr   # x-axis upper limit for ALL subplots
        # Frame times for the raw VAD probabilities (silero window = 512 samples).
        FRAME_N = 512
        frame_t1 = np.arange(len(ovl["probs1"])) * FRAME_N / sr
        frame_t2 = np.arange(len(ovl["probs2"])) * FRAME_N / sr

        # Mask application has moved to Stage 3c (post_separation_processing),
        # so 3b's entries don't carry `s_gated` anymore. Compute `raw * mask`
        # locally just for this diagnostic — it's what `backend: naive` will
        # produce in 3c, so the overlay still shows what the VAD mask does.
        s1_masked_preview = ovl["s1_raw"] * ovl["mask1"]
        s2_masked_preview = ovl["s2_raw"] * ovl["mask2"]

        # 6 subplots: narrow pyannote-diarization band on top, then the five
        # original ones (mix, s1, s2, masks, VAD probs).
        fig, all_axes = plt.subplots(
            6, 1, figsize=(14, 8.5), sharex=True,
            gridspec_kw={"height_ratios": [0.3, 1, 1, 1, 1, 1]},
        )
        diar_ax = all_axes[0]
        axes = all_axes[1:]   # keep the existing axes[0..4] indexing below

        # Pyannote segments restricted to this overlap's padded window. Each
        # segment is clipped to [0, window_dur] before drawing — pyannote
        # segments often extend beyond the padded window, and an unclipped
        # bar would stretch the (shared) x-axis past the actual window
        # bounds and shrink the audio plots below to fit.
        #
        # Speaker IDs are pyannote's own labels (e.g. "SPEAKER_00"); the
        # s1/s2 → pyannote-speaker mapping is established later by ECAPA in
        # Stage 4 — at 3b time it's not yet known. The two pyannote speakers
        # are coloured with C1/C2 to match the s1/s2 rows visually, but the
        # mapping is positional (sorted-label order), not a real assignment.
        if diar_df_full is not None and len(diar_df_full):
            in_window = ((diar_df_full["end"] > ovl["pad_start"])
                         & (diar_df_full["start"] < ovl["pad_end"]))
            diar_segs = diar_df_full.loc[in_window].copy()
            diar_segs["start"] = diar_segs["start"] - ovl["pad_start"]
            diar_segs["end"]   = diar_segs["end"]   - ovl["pad_start"]
            speakers = sorted(diar_segs["speaker"].unique().tolist())
        else:
            diar_segs = None
            speakers = []
        diar_colors = ["C1", "C2", "C3", "C4"]   # accommodate >2 if pyannote drifts
        for y, spk in enumerate(speakers):
            rows = diar_segs[diar_segs["speaker"] == spk]
            for _, r in rows.iterrows():
                bar_start = max(r["start"], 0.0)
                bar_end   = min(r["end"],   window_dur)
                bar_dur   = bar_end - bar_start
                if bar_dur <= 0:
                    continue
                diar_ax.barh(y, bar_dur, left=bar_start,
                             height=0.8, color=diar_colors[y % len(diar_colors)],
                             alpha=0.7)
        diar_ax.set_yticks(list(range(len(speakers))))
        diar_ax.set_yticklabels(speakers, fontsize=8)
        diar_ax.set_ylim(-0.6, max(len(speakers) - 0.4, 0.6))
        diar_ax.set_ylabel("pyannote", fontsize=9)

        axes[0].plot(t, ovl["mix"], lw=0.5, color="C0");  axes[0].set_ylabel("mix")
        axes[1].plot(t, ovl["s1_raw"], lw=0.5, color="C1", alpha=0.4, label="raw")
        axes[1].plot(t, s1_masked_preview, lw=0.5, color="C1", label="raw × mask")
        axes[1].set_ylabel("s1"); axes[1].legend(fontsize=8, loc="upper right")
        axes[2].plot(t, ovl["s2_raw"], lw=0.5, color="C2", alpha=0.4, label="raw")
        axes[2].plot(t, s2_masked_preview, lw=0.5, color="C2", label="raw × mask")
        axes[2].set_ylabel("s2"); axes[2].legend(fontsize=8, loc="upper right")
        axes[3].plot(t, ovl["mask1"], color="C1", label="mask 1")
        axes[3].plot(t, ovl["mask2"], color="C2", label="mask 2", alpha=0.7)
        axes[3].set_ylabel("mask"); axes[3].set_ylim(-0.1, 1.1)
        axes[3].legend(fontsize=8, loc="upper right")
        # Raw silero probabilities (step plot makes the 32 ms frame discretization
        # visible). Thresholds drawn as horizontal lines so you can read off
        # which frames are above/below the upper/soft-threshold cutoff.
        axes[4].step(frame_t1, ovl["probs1"], where="post", color="C1", label="probs 1")
        axes[4].step(frame_t2, ovl["probs2"], where="post", color="C2", label="probs 2", alpha=0.7)
        axes[4].axhline(cfg.separation.vad_threshold, color="k", ls="--", lw=0.7,
                        alpha=0.6, label=f"upper={cfg.separation.vad_threshold}")
        if 0 < cfg.separation.vad_soft_threshold < cfg.separation.vad_threshold:
            axes[4].axhline(cfg.separation.vad_soft_threshold, color="k", ls=":",
                            lw=0.7, alpha=0.6,
                            label=f"soft={cfg.separation.vad_soft_threshold}")
        axes[4].set_ylabel("VAD probs"); axes[4].set_ylim(-0.05, 1.05)
        axes[4].legend(fontsize=8, loc="upper right")
        axes[4].set_xlabel("s in padded window")
        # Belt-and-braces: lock x-axis to the actual window, in case a stray
        # plot call ever drags it outside [0, window_dur] again.
        diar_ax.set_xlim(0, window_dur)
        emit_lo = ovl["emit_start"] - ovl["pad_start"]
        emit_hi = ovl["emit_end"]   - ovl["pad_start"]
        # Dotted lines show pyannote's original overlap boundary — when
        # snap_to_silence is in use, the green emit region should sit
        # outside them.
        orig_lo = ovl["start"] - ovl["pad_start"]
        orig_hi = ovl["end"]   - ovl["pad_start"]
        for ax in all_axes:
            ax.axvspan(emit_lo, emit_hi, color="green", alpha=0.08)
            ax.axvline(orig_lo, color="black", ls=":", lw=0.7, alpha=0.5)
            ax.axvline(orig_hi, color="black", ls=":", lw=0.7, alpha=0.5)
            ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

        print("mix:");          display(Audio(ovl["mix"], rate=sr))
        print("s1 raw × mask:"); display(Audio(s1_masked_preview, rate=sr))
        print("s2 raw × mask:"); display(Audio(s2_masked_preview, rate=sr))


## Stage 3c — Post-separation processing (VAD mask + optional BWE)

This stage is the **only** place where the VAD mask gets applied. Stage 3b
computes the mask (it needs it for `snap_to_silence` emit-region decisions)
but no longer multiplies it in — that's done here, after the optional BWE
pass. Splitting compute-vs-apply lets BWE see the unmasked separator output,
avoiding the ringing artefacts that hard mask edges feed into a BWE model.

The stage **always runs** — Stage 4 depends on `s_gated` being populated.
To do "no BWE", set `backend: naive` — the stage still runs (applying the
mask) but no neural model touches the audio. The 8 kHz spectral content is
left as-is.

Knobs:

- `cfg.post_separation_processing.backend` — `naive` (mask only) | `ap_bwe` | `flowhigh`
- `cfg.post_separation_processing.checkpoint_path` — `ap_bwe` only
- `cfg.post_separation_processing.flowhigh_input_sr` — `flowhigh` only (`8000` | `16000`)

For each overlap region the cell plots spectrograms of pre- vs post-3c for
both streams. Both panels have the VAD mask applied, so the difference is
purely what the backend did to the spectrum — the 4-8 kHz band is empty in
the pre panel and (for `ap_bwe` / `flowhigh`) populated in the post panel.

Re-running this cell is safe: a snapshot of the pre-3c state (`raw × mask`)
is stashed onto `ctx` keyed on the current separation output, so subsequent
runs still have a clean reference to compare against. Re-running Stage 3b
invalidates the snapshot automatically.

In [ ]:
# --- Knobs ---
# cfg.post_separation_processing.backend          = "flowhigh"   # naive | ap_bwe | flowhigh
# cfg.post_separation_processing.flowhigh_input_sr = 8000
# cfg.post_separation_processing.checkpoint_path  = "/home/user/AP-BWE/checkpoints/8kto16k/g_8kto16k.zip"

# Inspection cap. Each overlap produces one 2×2 spectrogram figure + four
# audio players (s1 pre/post, s2 pre/post) — heavier per item than Stage
# 3b's outputs, so the cap is lower. Same WSL-bridge concern: too many
# inline outputs in a single cell crashes the notebook view. A one-line
# summary is printed for every overlap; only the first MAX_DETAILED_PLOTS
# get the full spectrogram + audio treatment.
MAX_DETAILED_PLOTS = 20

# Stash the pre-3c state (`raw × mask` — what `backend: naive` would write)
# the first time this cell runs, so re-runs still have a clean before/after
# to compare even after `s_gated` has been overwritten. Keyed on the current
# separation-output identity so that re-running Stage 3b invalidates the
# snapshot automatically.
if ctx.overlap_separated:
    sep_key = id(ctx.overlap_separated)
    if getattr(ctx, "_sr_pre_snapshot_key", None) != sep_key:
        ctx._sr_pre_snapshot = [
            ((e["s1_raw"] * e["mask1"]).astype(np.float32),
             (e["s2_raw"] * e["mask2"]).astype(np.float32))
            for e in ctx.overlap_separated
        ]
        ctx._sr_pre_snapshot_key = sep_key

pipeline.run_stage("post_separation_processing", ctx)


# --- Inspect ---
sr = ctx.sample_rate
print(f"backend         : {cfg.post_separation_processing.backend}")
print(f"checkpoint_path : {cfg.post_separation_processing.checkpoint_path}")

if not ctx.overlap_separated:
    print("No overlap regions in this recording.")
else:
    # 1) Compact summary for every overlap (cheap, always shown).
    n_total = len(ctx.overlap_separated)
    print(f"\n{n_total} overlap region(s):")
    print(f"  {'idx':>4} {'orig (s)':>16}  {'dur':>6}  {'pre RMS':>9}  {'post RMS':>9}")
    for i_ovl, ovl in enumerate(ctx.overlap_separated):
        s1_pre, s2_pre = ctx._sr_pre_snapshot[i_ovl]
        s1_post, s2_post = ovl["s1_gated"], ovl["s2_gated"]
        pre_rms = float(np.sqrt(np.mean(s1_pre.astype(np.float64) ** 2) + 1e-12))
        post_rms = float(np.sqrt(np.mean(s1_post.astype(np.float64) ** 2) + 1e-12))
        dur = ovl["end"] - ovl["start"]
        print(f"  #{ovl['idx']:>3} "
              f"[{ovl['start']:6.2f},{ovl['end']:6.2f}]  "
              f"{dur:5.2f}s  {pre_rms:9.5f}  {post_rms:9.5f}")

    # 2) Detailed plot + audio only for the first MAX_DETAILED_PLOTS.
    to_render = list(enumerate(ctx.overlap_separated))[:MAX_DETAILED_PLOTS]
    if n_total > len(to_render):
        print(f"\nRendering plots + audio for first {len(to_render)} of {n_total} "
              f"overlaps (raise MAX_DETAILED_PLOTS to see more).")

    nfft = 512
    for i_ovl, ovl in to_render:
        s1_pre, s2_pre = ctx._sr_pre_snapshot[i_ovl]
        s1_post, s2_post = ovl["s1_gated"], ovl["s2_gated"]

        print(f"\nOverlap #{ovl['idx']} — orig=[{ovl['start']:.2f}, {ovl['end']:.2f}]")

        # 2×2 spectrograms: rows = pre/post, cols = s1/s2. Linear-frequency
        # axis up to Nyquist (8 kHz at sr=16k) so the empty 4–8 kHz band in
        # the pre row is visually obvious next to the populated post row.
        fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharex=True, sharey=True)
        for col, (label, pre, post) in enumerate([("s1", s1_pre, s1_post),
                                                   ("s2", s2_pre, s2_post)]):
            for row, (tag, audio) in enumerate([("pre-3c", pre), ("post-3c", post)]):
                ax = axes[row, col]
                if audio.size > nfft:
                    ax.specgram(audio, NFFT=nfft, Fs=sr, noverlap=nfft // 2,
                                cmap="magma", scale="dB", vmin=-90, vmax=-20)
                ax.set_title(f"{label} {tag}", fontsize=10)
                ax.set_ylim(0, sr / 2)
                if row == 1:
                    ax.set_xlabel("s")
                if col == 0:
                    ax.set_ylabel("Hz")
        plt.tight_layout(); plt.show()

        print("s1 pre-3c :"); display(Audio(s1_pre,  rate=sr))
        print("s1 post-3c:"); display(Audio(s1_post, rate=sr))
        print("s2 pre-3c :"); display(Audio(s2_pre,  rate=sr))
        print("s2 post-3c:"); display(Audio(s2_post, rate=sr))

## Stage 4 — Per-speaker assembly

Stream view in original recording time. Blue = solo (enhanced), red = overlap
(separated). Audio players play each assembled per-speaker stream.

In [ ]:
# --- Knobs ---
# cfg.assembly.output_mode              = "full_length"   # shortened | full_length
# cfg.assembly.silence_separator_s      = 0.3
# cfg.assembly.crossfade_ms             = 5.0             # internal piece-to-piece seam fade
# cfg.assembly.edge_fade_ms             = 2.0             # stream's outermost edge fade
# cfg.assembly.overlap_rms_match_solo   = True            # scale overlap pieces to per-speaker solo RMS
# cfg.assembly.anchor_max_duration_s    = 120.0            # cap solo audio fed to ECAPA (None = no cap)
# cfg.assembly.per_piece_rms_norm     = True            # aggressive: normalise ALL pieces (overrides above)
# cfg.assembly.target_rms             = 0.05

pipeline.run_stage("assembly", ctx)


# --- Inspect ---
sr = ctx.sample_rate
spk_to_label = ctx.spk_to_label

fig, axes = plt.subplots(
    len(ctx.speakers), 1,
    figsize=(16, 1.2 * len(ctx.speakers) + 1),
    sharex=True, squeeze=False,
)
axes = axes[:, 0]
for ax, spk in zip(axes, ctx.speakers):
    for entry in ctx.timestamp_map.per_speaker[spk]:
        color = "#1f77b4" if entry.kind == "solo" else "#e74c3c"
        ax.barh(0, entry.orig_end - entry.orig_start, left=entry.orig_start,
                height=0.6, color=color, alpha=0.85)
    ax.set_yticks([0])
    ax.set_yticklabels([f"{spk_to_label[spk]}: {spk}"])
    ax.set_ylim(-0.5, 0.5)
    ax.grid(axis="x", alpha=0.3)
axes[-1].set_xlabel("orig time (s)")
axes[-1].set_xlim(0, ctx.diarization.total_duration_s)
axes[0].set_title("Assembled streams in original time  (blue = solo / SE, red = overlap / SS)")
plt.tight_layout(); plt.show()

print(f"output_mode  : {cfg.assembly.output_mode}")
print(f"weak_anchor  : {ctx.weak_anchor}")
for spk, audio in ctx.assembled.items():
    print(f"\nSpeaker {spk_to_label[spk]} ({spk}) — {len(audio)/sr:.2f}s")
    display(Audio(audio, rate=sr))

## Stage 5 — Transcripts

One Whisper call per assembled per-speaker stream. Word timestamps are
available in `ctx.transcripts[spk]['segments'][i]['words']`.

In [ ]:
# --- Knobs ---
# Backend = engine selector. Two options:
#   "whisper"  — openai-whisper, OpenAI checkpoints only (large-v3, large-v2, …),
#                approximate word timestamps (drift 200–500 ms).
#   "whisperx" — faster-whisper + wav2vec2 forced alignment. Accepts any HF
#                Whisper id (e.g. a Polish-finetuned Whisper). Accurate word
#                timestamps (~50 ms). Loads TWO models: the Whisper backbone
#                AND the wav2vec2 aligner.
cfg.transcription.backend           = "whisperx"
# Whisper backbone. With backend: whisperx this can be any HF Whisper id.
#   (any HF Whisper finetune is accepted by the whisperx backend)
#   Vanilla OpenAI:                       large-v3, large-v2
cfg.transcription.model_name        = "large-v2"  # or large-v3
# wav2vec2 forced-alignment model. ONLY used when backend == whisperx.
# Ignored otherwise.
cfg.transcription.align_model_name  = "jonatasgrosman/wav2vec2-large-xlsr-53-polish"
cfg.transcription.language          = "pl"
cfg.transcription.initial_prompt    = "Rozmowa po polsku."
cfg.transcription.word_timestamps   = True

pipeline.run_stage("transcription", ctx)


# --- Inspect ---
spk_to_label = ctx.spk_to_label
print(f"backend          : {cfg.transcription.backend}")
print(f"model_name       : {cfg.transcription.model_name}")
if cfg.transcription.backend == "whisperx":
    print(f"align_model_name : {cfg.transcription.align_model_name}")
print()

for spk in ctx.speakers:
    label = spk_to_label[spk]
    print(f"\n=== Speaker {label} ({spk}) ===")
    res = ctx.transcripts.get(spk, {})
    segments = res.get("segments", [])
    if not segments:
        text = res.get("text", "")
        print(text if text else "(empty)")
        continue
    for seg in segments:
        print(f"[{seg['start']:6.2f} → {seg['end']:6.2f}]  {seg['text'].strip()}")

## Save outputs to disk

Materialise the per-recording layout the eval module expects:
`<OUTPUT_DIR>/pipeline/{diarization.json, routing.json, stream_A.wav, stream_B.wav, transcript_A.{txt,json}, transcript_B.{txt,json}, transcript_mixture.{txt,json} (if enabled), metadata.json}`.

Transcripts use decimal-seconds format, no speaker headers — compatible with `parse_gt_txt`.

In [ ]:
# --- Save pipeline outputs ---
from asr_pipeline.io import write_pipeline_outputs
from dataclasses import asdict

DATASET = "clarin"   # clarin | libricss | ...  used as the per-dataset subdir under the eval tree
recording_id = ctx.input_path.stem if ctx.input_path else "unknown"
OUTPUT_DIR = Path.home() / "datasets" / "eval" / DATASET / recording_id

out = write_pipeline_outputs(ctx, OUTPUT_DIR, config_snapshot=asdict(cfg))
print(f"wrote pipeline outputs to {out}")
for p in sorted(out.iterdir()):
    print(f"  {p.name:30s} ({p.stat().st_size:>10d} bytes)")

## Iteration tips

- **Re-run one stage.** Edit the knob lines at the top of the stage's cell,
  then re-run that cell. Re-running the same stage twice is fast — the
  stage's model stays loaded.
- **Switch stages.** Just run a different stage's cell. The previous stage's
  model is freed, the new one loads. Only one model on GPU at any time.
- **Go back.** If you want to re-run an upstream stage after changing its
  knobs, just rerun its cell. Then re-run any downstream stages whose inputs
  changed.
- **Free GPU memory.** Run the final cell (`pipeline.unload()`) to free
  whatever's currently loaded. Useful before swapping checkpoints.
- **Reload after editing checkpoint paths.** If you change
  `cfg.<stage>.checkpoint_path`, call `pipeline.unload()` first — the loaded
  model is the old one. After unload, the next `run_stage` call will reload.
- **Persist artefacts.** Set `cfg.artifact_dir = "runs/notebook"` and
  `cfg.spill_intermediate = True` in cell 3 (then re-create the pipeline /
  re-run the stage) to also write each stage's outputs to disk.

In [ ]:
# Free whatever's currently loaded on the GPU.
pipeline.unload()
print(pipeline)